# Week 14 Companion Notebook — SoW, Pricing & Solution Architecture

**The Meridian Engagement · "Paper before product"**

This is the deep reference behind the lean slides. Pure Python / stdlib — runs top to
bottom with no installs and no edits. It contains five calculators plus a full glossary:

1. **Estimation / T-shirt-sizing calculator** with per-risk-class buffers
2. **3-way pricing calculator** (T&M-cap vs fixed vs outcome-based) that prices the Meridian pilot and compares margin + risk
3. **SoW template** generator (acceptance criteria tied to the W13/M0 eval anchors, IP, data annex)
4. **Stack-choice decision matrix** as a scored table (incl. when Azure / Google stacks win)
5. **Integration-topology selector** (P2P / iPaaS / event mesh)

Every named term from the slides is defined in the **Glossary** at the end.

In [1]:
# Stdlib only.
from dataclasses import dataclass, field
from typing import Optional

def money(cents_or_dollars, is_cents=False):
    v = cents_or_dollars / 100 if is_cents else cents_or_dollars
    return f"${v:,.0f}"

def rule(title=""):
    line = "=" * 64
    print(line)
    if title:
        print(title)
        print(line)

print("Week 14 notebook ready. Stdlib only, no installs needed.")

Week 14 notebook ready. Stdlib only, no installs needed.


## 1. Estimation & T-shirt sizing with risk buffers

FDEs never quote a single number. We decompose into **milestones**, size each with a
**T-shirt size** (S/M/L/XL → a day *range*, not a point), and apply a **risk buffer** that
is *per risk class*, not one flat 20%. The output is a **low / expected / high** engagement
estimate with the math shown — a number you can defend to a board line by line.

The biggest buffer at a regulated client is the **data-access / controls-review** risk —
the Palantir reality: historically *8–12 weeks of data-access politics per week of value demo*
(Nabeel Qureshi, *Reflections on Palantir*, 2023). Price the politics or eat them.

In [2]:
# T-shirt size -> (low_days, high_days). Ranges, because nobody can defend "37 hours".
TSHIRT = {
    "S":  (3, 5),
    "M":  (8, 12),
    "L":  (15, 25),
    "XL": (30, 50),
}

# Risk class -> buffer multiplier on the HIGH end. Access risk dominates at a bank.
RISK_BUFFER = {
    "low":            1.10,   # well-understood work (UI, plumbing you've done before)
    "integration":    1.35,   # legacy systems, rate limits, undocumented APIs
    "access_politics": 1.80,  # the controls-review / data-access gate (the Palantir pole)
    "novel":          1.50,   # genuinely new modeling / unproven approach
}

@dataclass
class Milestone:
    name: str
    size: str          # S / M / L / XL
    risk_class: str    # key into RISK_BUFFER

def estimate(milestones, blended_day_rate):
    rows = []
    tot_low = tot_exp = tot_high = 0.0
    for m in milestones:
        low, high = TSHIRT[m.size]
        buf = RISK_BUFFER[m.risk_class]
        exp = (low + high) / 2
        high_buffered = high * buf            # buffer applied to the high end
        tot_low += low
        tot_exp += exp
        tot_high += high_buffered
        rows.append((m.name, m.size, m.risk_class, low, exp, round(high_buffered, 1), buf))
    return rows, tot_low, tot_exp, tot_high, blended_day_rate

# The Meridian pilot, decomposed.
MERIDIAN_MILESTONES = [
    Milestone("M1 SoW + architecture pack",            "S",  "low"),
    Milestone("Data-access / controls review (the gate)","M", "access_politics"),
    Milestone("M2 Triage+Evidence+Policy agents in CI", "L",  "novel"),
    Milestone("M3 MCP integration + RBAC + audit",      "L",  "integration"),
    Milestone("M4 Eval gate + dashboards + handover",   "M",  "novel"),
]

BLENDED_DAY_RATE = 1800  # USD/day, blended FDE team rate (illustrative)

rows, lo, exp, hi, rate = estimate(MERIDIAN_MILESTONES, BLENDED_DAY_RATE)

rule("ESTIMATION — Meridian pilot")
hdr = f"{'Milestone':38} {'Sz':3} {'RiskClass':16} {'low':>5} {'exp':>6} {'high*':>7} {'buf':>5}"
print(hdr)
print("-" * len(hdr))
for name, size, rc, l, e, h, buf in rows:
    print(f"{name:38.38} {size:3} {rc:16} {l:5} {e:6.1f} {h:7.1f} {buf:5.2f}")
print("-" * len(hdr))
print(f"{'TOTAL (days)':38} {'':3} {'':16} {lo:5.0f} {exp:6.1f} {hi:7.1f}")
print()
print(f"Blended day rate: {money(rate)}/day")
print(f"  Low   estimate: {money(lo*rate)}  ({lo:.0f} days)")
print(f"  Expected       : {money(exp*rate)}  ({exp:.0f} days)")
print(f"  High (buffered): {money(hi*rate)}  ({hi:.0f} days)")
print()
print("* high end carries the per-risk-class buffer. The access-politics line")
print("  carries the biggest buffer (1.80x) — that's the board-defensible honesty.")

ESTIMATION — Meridian pilot
Milestone                              Sz  RiskClass          low    exp   high*   buf
--------------------------------------------------------------------------------------
M1 SoW + architecture pack             S   low                  3    4.0     5.5  1.10
Data-access / controls review (the gat M   access_politics      8   10.0    21.6  1.80
M2 Triage+Evidence+Policy agents in CI L   novel               15   20.0    37.5  1.50
M3 MCP integration + RBAC + audit      L   integration         15   20.0    33.8  1.35
M4 Eval gate + dashboards + handover   M   novel                8   10.0    18.0  1.50
--------------------------------------------------------------------------------------
TOTAL (days)                                                   49   64.0   116.3

Blended day rate: $1,800/day
  Low   estimate: $88,200  (49 days)
  Expected       : $115,200  (64 days)
  High (buffered): $209,430  (116 days)

* high end carries the per-risk-class buffer. Th

### What to tell the CFO
Quote the **range with named drivers**, not a point. "Expected ~X, but the controls-review
gate could push it to ~Y; that line carries our biggest buffer because data access is the
longest pole at a regulated bank. We've priced the politics so you're not surprised in month two."

In [3]:
# Sensitivity: what if the access-politics gate is worse than feared? Re-run at 2.2x.
RISK_BUFFER_PESSIMISTIC = dict(RISK_BUFFER, access_politics=2.20)
def estimate_with(buffers):
    tot_high = 0.0
    for m in MERIDIAN_MILESTONES:
        _, high = TSHIRT[m.size]
        tot_high += high * buffers[m.risk_class]
    return tot_high

base_high = estimate_with(RISK_BUFFER)
pess_high = estimate_with(RISK_BUFFER_PESSIMISTIC)
print(f"High estimate, base access buffer (1.80x): {base_high:.0f} days = {money(base_high*rate)}")
print(f"High estimate, pessimistic access (2.20x): {pess_high:.0f} days = {money(pess_high*rate)}")
print(f"Delta from the access-politics assumption alone: {money((pess_high-base_high)*rate)}")
print("Lesson: at a regulated client the access assumption moves the number more than the code does.")

High estimate, base access buffer (1.80x): 116 days = $209,430
High estimate, pessimistic access (2.20x): 121 days = $218,070
Delta from the access-politics assumption alone: $8,640
Lesson: at a regulated client the access assumption moves the number more than the code does.


## 2. Three-way pricing calculator

Same scope, three commercial wrappers. The model decides **who holds the risk** and **your margin profile**.

| Model | You bill on | Risk sits with | Margin |
|---|---|---|---|
| **T&M with a cap** | hours, ceiling-bounded | shared (client caps exposure) | predictable, capped upside |
| **Fixed-price** | a scoped deliverable | **vendor** (you eat overruns) | high if scoped well, brutal if not |
| **Outcome-based** | a business result | shared, results-linked | highest upside, highest variance |

**Live cases (verifiable):**
- **Sierra** (Bret Taylor / Clay Bavor) prices on the **outcome — the resolution**, not seats; reported **$150M+ ARR**. The trap is the definition of "resolved." (Sierra public statements & press, 2024–2025.)
- **Palantir** runs **acquire-expand-scale**; Airbus → **Skywise** became a product. Early pilots are priced to survive the access fight; expansion is where economics work. (Nabeel Qureshi, *Reflections on Palantir*, 2023.)

In [4]:
# Shared cost basis for the pilot (so margins are comparable across models).
@dataclass
class CostBasis:
    expected_days: float
    blended_day_rate: float
    fixed_overhead: float = 12000.0   # tooling, infra, PM, eval-set authoring
    @property
    def expected_cost(self):
        return self.expected_days * self.blended_day_rate + self.fixed_overhead

basis = CostBasis(expected_days=exp, blended_day_rate=rate)
print(f"Expected delivery cost (the floor under every model): {money(basis.expected_cost)}")
print(f"  = {exp:.0f} days x {money(rate)} + {money(basis.fixed_overhead)} overhead")

Expected delivery cost (the floor under every model): $127,200
  = 64 days x $1,800 + $12,000 overhead


In [5]:
def price_tm_cap(basis, markup=1.35, cap_multiple=1.25):
    """Bill cost x markup on hours, ceiling-bounded at cap_multiple of expected price."""
    expected_price = basis.expected_cost * markup
    cap = expected_price * cap_multiple
    margin = expected_price - basis.expected_cost
    return {
        "model": "T&M with a cap",
        "expected_revenue": expected_price,
        "max_revenue": cap,                       # client's ceiling / your max
        "min_revenue": basis.expected_cost * markup * 0.6,  # if it finishes early
        "expected_margin": margin,
        "margin_pct": margin / expected_price,
        "risk_to_vendor": "LOW — overruns billed up to the cap; only risk is finishing under-budget",
    }

def price_fixed(basis, target_margin_pct=0.30):
    """One number. Vendor eats all overruns; keeps all the upside of running lean."""
    price = basis.expected_cost / (1 - target_margin_pct)
    # If actuals come in 25% over expected, margin erodes:
    cost_if_overrun = basis.expected_cost * 1.25
    margin_if_overrun = price - cost_if_overrun
    return {
        "model": "Fixed-price",
        "expected_revenue": price,
        "max_revenue": price,                     # fixed: no upside above the number
        "min_revenue": price,                     # you still get paid the number...
        "expected_margin": price - basis.expected_cost,
        "margin_pct": (price - basis.expected_cost) / price,
        "margin_if_25pct_overrun": margin_if_overrun,
        "risk_to_vendor": "HIGH — a 25% overrun eats your margin; discovery quality IS the hedge",
    }

def price_outcome(basis, floor_margin_pct=0.10, bonus_per_resolution=18.0,
                  expected_resolutions=4000, low_resolutions=1500, high_resolutions=7000):
    """Fixed floor + bonus per dispute auto-resolved within Reg E. Aligned, variable."""
    floor = basis.expected_cost / (1 - floor_margin_pct)   # thin floor, covers cost + a little
    def rev(n): return floor + n * bonus_per_resolution
    expected_rev = rev(expected_resolutions)
    return {
        "model": "Outcome-based (floor + bonus/resolution)",
        "expected_revenue": expected_rev,
        "max_revenue": rev(high_resolutions),
        "min_revenue": rev(low_resolutions),
        "expected_margin": expected_rev - basis.expected_cost,
        "margin_pct": (expected_rev - basis.expected_cost) / expected_rev,
        "bonus_basis": f"{money(bonus_per_resolution)} per dispute auto-resolved within Reg E",
        "who_counts": "eval suite + audit log over the agreed golden/live set, reviewed monthly",
        "risk_to_vendor": "HIGH VARIANCE — upside is real but depends on a number the CISO must let you measure",
    }

quotes = [
    price_tm_cap(basis),
    price_fixed(basis),
    price_outcome(basis),
]

rule("PRICING — Meridian pilot, three ways")
hdr = f"{'Model':40} {'expected':>12} {'min':>12} {'max':>12} {'margin%':>8}"
print(hdr); print("-" * len(hdr))
for q in quotes:
    print(f"{q['model']:40.40} {money(q['expected_revenue']):>12} "
          f"{money(q['min_revenue']):>12} {money(q['max_revenue']):>12} "
          f"{q['margin_pct']*100:7.1f}%")
print("-" * len(hdr))
for q in quotes:
    print(f"\n{q['model']}")
    print(f"  risk to vendor : {q['risk_to_vendor']}")
    if "margin_if_25pct_overrun" in q:
        print(f"  margin if +25% overrun: {money(q['margin_if_25pct_overrun'])} (vs expected {money(q['expected_margin'])})")
    if "bonus_basis" in q:
        print(f"  bonus basis    : {q['bonus_basis']}")
        print(f"  who counts     : {q['who_counts']}")

PRICING — Meridian pilot, three ways
Model                                        expected          min          max  margin%
----------------------------------------------------------------------------------------
T&M with a cap                               $171,720     $103,032     $214,650    25.9%
Fixed-price                                  $181,714     $181,714     $181,714    30.0%
Outcome-based (floor + bonus/resolution)     $213,333     $168,333     $267,333    40.4%
----------------------------------------------------------------------------------------

T&M with a cap
  risk to vendor : LOW — overruns billed up to the cap; only risk is finishing under-budget

Fixed-price
  risk to vendor : HIGH — a 25% overrun eats your margin; discovery quality IS the hedge
  margin if +25% overrun: $22,714 (vs expected $54,514)

Outcome-based (floor + bonus/resolution)
  risk to vendor : HIGH VARIANCE — upside is real but depends on a number the CISO must let you measure
  bonus basis    

### Recommendation for a first pilot with a nervous regulated client
**Fixed-price floor + outcome bonus.** The fixed floor gives procurement the *ceiling* they
demanded and protects you while access politics resolve; the outcome bonus gives the CIO the
*alignment* story for the board. It's the Sierra idea (price on the resolution) wrapped in the
Palantir reality (the early deal must survive the access fight first).

In [6]:
# Combine: fixed floor that covers cost+ with a capped outcome bonus on top.
def price_floor_plus_bonus(basis, floor_margin_pct=0.20, bonus_per_resolution=18.0,
                           expected_resolutions=4000, bonus_cap_resolutions=6000):
    floor = basis.expected_cost / (1 - floor_margin_pct)
    capped = min(expected_resolutions, bonus_cap_resolutions)
    expected_rev = floor + capped * bonus_per_resolution
    max_rev = floor + bonus_cap_resolutions * bonus_per_resolution
    return {
        "floor": floor,
        "expected_revenue": expected_rev,
        "max_revenue": max_rev,
        "expected_margin_pct": (expected_rev - basis.expected_cost) / expected_rev,
        "bonus_cap": f"{bonus_cap_resolutions} resolutions @ {money(bonus_per_resolution)}",
    }

rec = price_floor_plus_bonus(basis)
rule("RECOMMENDED — fixed floor + capped outcome bonus")
print(f"  Floor (procurement's ceiling on the base): {money(rec['floor'])}")
print(f"  Expected revenue : {money(rec['expected_revenue'])}")
print(f"  Max revenue      : {money(rec['max_revenue'])}  (bonus capped at {rec['bonus_cap']})")
print(f"  Expected margin  : {rec['expected_margin_pct']*100:.1f}%")
print("  Why: floor protects you through the controls-review gate; bonus aligns you with the CIO's board metric.")

RECOMMENDED — fixed floor + capped outcome bonus
  Floor (procurement's ceiling on the base): $159,000
  Expected revenue : $231,000
  Max revenue      : $267,000  (bonus capped at 6000 resolutions @ $18)
  Expected margin  : 44.9%
  Why: floor protects you through the controls-review gate; bonus aligns you with the CIO's board metric.


## 3. SoW template generator

Acceptance criteria are the **M0 eval anchors** (that's the through-line the whole engagement
is graded on). The generator below renders an SoW skeleton from structured inputs so the
*acceptance criteria are literally the success criteria you scoped at discovery* — plus IP terms
and a data annex reflecting the synthetic-data-only constraint.

In [7]:
@dataclass
class EvalAnchor:
    """An M0 success criterion, becomes an SoW acceptance clause verbatim."""
    description: str
    threshold: str

@dataclass
class SoWInputs:
    pricing_model: str
    eval_anchors: list
    ip_owner_agents: str = "Client"
    ip_owner_prompts: str = "Client"
    ip_owner_finetunes: str = "none created in pilot"
    ip_owner_embeddings: str = "Client (deleted on exit)"
    data_environment: str = "Synthetic data only until controls review passes"
    subprocessors: tuple = ("AWS Bedrock (Anthropic Claude — model inference in the client's AWS account)",)

MERIDIAN_ANCHORS = [
    EvalAnchor("Type-A disputes auto-triaged with policy-citation accuracy", ">=80% triaged, >=95% citation accuracy"),
    EvalAnchor("Reg E clock breaches introduced by the agent", "= 0"),
    EvalAnchor("Provisional-credit recommendations cite a policy chunk (output tripwire)", ">=99%"),
    EvalAnchor("Two-analyst RBAC test returns correctly-scoped results + matching audit", "passes for both accounts"),
]

def render_sow(inp: SoWInputs) -> str:
    out = []
    out.append("STATEMENT OF WORK — Meridian Dispute-Resolution Agent Pilot")
    out.append("Vendor: Forgepoint AI   Client: Meridian Bank")
    out.append("")
    out.append("§4 ACCEPTANCE CRITERIA  (= the M0 eval anchors)")
    for i, a in enumerate(inp.eval_anchors, 1):
        out.append(f"  {i}. Accepted when: {a.description} -> {a.threshold}")
    out.append("")
    out.append("§5 PRICING")
    out.append(f"  Model: {inp.pricing_model}")
    out.append("")
    out.append("§7 IP TERMS")
    out.append(f"  Agents/orchestration : {inp.ip_owner_agents}")
    out.append(f"  Prompts              : {inp.ip_owner_prompts}")
    out.append(f"  Fine-tunes/weights   : {inp.ip_owner_finetunes}")
    out.append(f"  Embeddings/derived   : {inp.ip_owner_embeddings}")
    out.append("")
    out.append("§8 DATA ANNEX")
    out.append(f"  Data environment     : {inp.data_environment}")
    out.append(f"  Subprocessors        : {', '.join(inp.subprocessors)}")
    out.append("  Audit                : every agent tool call logged immutably (identity, ts, allow/deny)")
    out.append("  Deletion on exit     : derived data deleted; deletion verified")
    out.append("")
    out.append("§9 SECURITY & AUTONOMY")
    out.append("  Agents act under analyst identity via OAuth2 on-behalf-of; no standing access.")
    out.append("  RBAC enforced in the MCP/integration layer (NOT the model).")
    out.append("  Money-moving actions (e.g., provisional credit) require human approval.")
    return "\n".join(out)

sow = render_sow(SoWInputs(pricing_model="Fixed-price floor + outcome bonus", eval_anchors=MERIDIAN_ANCHORS))
rule("GENERATED SOW SKELETON")
print(sow)

GENERATED SOW SKELETON
STATEMENT OF WORK — Meridian Dispute-Resolution Agent Pilot
Vendor: Forgepoint AI   Client: Meridian Bank

§4 ACCEPTANCE CRITERIA  (= the M0 eval anchors)
  1. Accepted when: Type-A disputes auto-triaged with policy-citation accuracy -> >=80% triaged, >=95% citation accuracy
  2. Accepted when: Reg E clock breaches introduced by the agent -> = 0
  3. Accepted when: Provisional-credit recommendations cite a policy chunk (output tripwire) -> >=99%
  4. Accepted when: Two-analyst RBAC test returns correctly-scoped results + matching audit -> passes for both accounts

§5 PRICING
  Model: Fixed-price floor + outcome bonus

§7 IP TERMS
  Agents/orchestration : Client
  Prompts              : Client
  Fine-tunes/weights   : none created in pilot
  Embeddings/derived   : Client (deleted on exit)

§8 DATA ANNEX
  Data environment     : Synthetic data only until controls review passes
  Subprocessors        : AWS Bedrock (Anthropic Claude — model inference in the client's 

The full prose template (for the graded M1 artifact) is `lab/sow-template.md`. This generator
exists to make the through-line mechanical: change your M0 anchors, the acceptance criteria
regenerate. If a criterion isn't measurable by an eval run, it doesn't belong here.

## 4. Stack-choice decision matrix (scored)

**Stack follows the client, not your preference.** We score the three candidate stacks across
weighted dimensions for *Meridian specifically* (an AWS-estate bank — core workloads, IAM and
compliance tooling already on AWS), then show the **conditions under which the answer flips**
to Azure or Google.


In [8]:
# Dimensions with weights (sum doesn't need to be 1; we normalize).
DIMENSIONS = {
    "client_platform_fit":   0.30,   # what the client already runs
    "enterprise_identity":   0.20,   # native RBAC / agent identity story
    "compliance_posture":    0.20,   # SOC2/GLBA maturity, isolation
    "cost":                  0.10,   # token + infra economics
    "ecosystem_maturity":    0.10,   # SDK, tooling, JD signal
    "in_vpc_isolation":      0.10,   # can it run inside the client's network boundary
}

# Scores 1-5 for Meridian (an AWS-estate bank).
STACKS = {
    "OpenAI Agents SDK + AWS Bedrock": {
        "client_platform_fit": 5, "enterprise_identity": 4, "compliance_posture": 5,
        "cost": 4, "ecosystem_maturity": 5, "in_vpc_isolation": 5,
    },
    "OpenAI Agents SDK + Azure Foundry": {
        "client_platform_fit": 2, "enterprise_identity": 5, "compliance_posture": 5,
        "cost": 3, "ecosystem_maturity": 5, "in_vpc_isolation": 4,
    },
    "Google ADK + Gemini + Vercel": {
        "client_platform_fit": 2, "enterprise_identity": 3, "compliance_posture": 4,
        "cost": 5, "ecosystem_maturity": 4, "in_vpc_isolation": 3,
    },
}

def score(stacks, dims):
    wsum = sum(dims.values())
    results = {}
    for name, sc in stacks.items():
        total = sum(sc[d] * w for d, w in dims.items()) / wsum
        results[name] = total
    return results

scores = score(STACKS, DIMENSIONS)

rule("STACK-CHOICE MATRIX — scored for Meridian (AWS-estate bank)")
dim_names = list(DIMENSIONS.keys())
hdr = f"{'Stack':34}" + "".join(f"{d[:9]:>10}" for d in dim_names) + f"{'WTD':>8}"
print(hdr); print("-" * len(hdr))
for name, sc in STACKS.items():
    row = f"{name:34.34}" + "".join(f"{sc[d]:>10}" for d in dim_names) + f"{scores[name]:>8.2f}"
    print(row)
print("-" * len(hdr))
print("weights: " + ", ".join(f"{d}={w}" for d, w in DIMENSIONS.items()))
winner = max(scores, key=scores.get)
print(f"\nWinner for Meridian: {winner}  ({scores[winner]:.2f})")
print("(the harness — OpenAI Agents SDK — is common to the top two: the SDK is portable; the CLOUD follows the client)")


STACK-CHOICE MATRIX — scored for Meridian (AWS-estate bank)
Stack                              client_pl enterpris complianc      cost ecosystem in_vpc_is     WTD
------------------------------------------------------------------------------------------------------
OpenAI Agents SDK + AWS Bedrock            5         4         5         4         5         5    4.70
OpenAI Agents SDK + Azure Foundry          2         5         5         3         5         4    3.80
Google ADK + Gemini + Vercel               2         3         4         5         4         3    3.20
------------------------------------------------------------------------------------------------------
weights: client_platform_fit=0.3, enterprise_identity=0.2, compliance_posture=0.2, cost=0.1, ecosystem_maturity=0.1, in_vpc_isolation=0.1

Winner for Meridian: OpenAI Agents SDK + AWS Bedrock  (4.70)
(the harness — OpenAI Agents SDK — is common to the top two: the SDK is portable; the CLOUD follows the client)


In [9]:
# When the answer FLIPS — re-score for a different client and watch the winner change.
def winner_line(label, stacks, dims, tag=""):
    s = score(stacks, dims)
    w = max(s, key=s.get)
    print(f"{label:44} -> {w}  ({s[w]:.2f}){tag}")

print("Same candidate stacks, different clients:")
winner_line("Meridian (AWS-estate bank)", STACKS, DIMENSIONS)

# M365/Entra-first enterprise: platform fit moves to Azure
STACKS_MS = {
    "OpenAI Agents SDK + AWS Bedrock":    dict(STACKS["OpenAI Agents SDK + AWS Bedrock"],    client_platform_fit=2),
    "OpenAI Agents SDK + Azure Foundry":  dict(STACKS["OpenAI Agents SDK + Azure Foundry"], client_platform_fit=5),
    "Google ADK + Gemini + Vercel":       dict(STACKS["Google ADK + Gemini + Vercel"],      client_platform_fit=2),
}
winner_line("M365/Entra-first enterprise", STACKS_MS, DIMENSIONS, "   <- Azure wins")

# GCP-native, cost-sensitive client: platform fit moves to Google, cost weighs more
STACKS_GCP = {
    "OpenAI Agents SDK + AWS Bedrock":    dict(STACKS["OpenAI Agents SDK + AWS Bedrock"],    client_platform_fit=2),
    "OpenAI Agents SDK + Azure Foundry":  dict(STACKS["OpenAI Agents SDK + Azure Foundry"], client_platform_fit=2),
    "Google ADK + Gemini + Vercel":       dict(STACKS["Google ADK + Gemini + Vercel"],      client_platform_fit=5),
}
gcp_weights = dict(DIMENSIONS, cost=0.25)
winner_line("GCP-native, cost-sensitive client", STACKS_GCP, gcp_weights, "   <- Google wins")

print("\nThe lesson: the matrix isn't to find THE answer; it's to make the client-fit logic explicit.")


Same candidate stacks, different clients:
Meridian (AWS-estate bank)                   -> OpenAI Agents SDK + AWS Bedrock  (4.70)
M365/Entra-first enterprise                  -> OpenAI Agents SDK + Azure Foundry  (4.70)   <- Azure wins
GCP-native, cost-sensitive client            -> Google ADK + Gemini + Vercel  (4.22)   <- Google wins

The lesson: the matrix isn't to find THE answer; it's to make the client-fit logic explicit.


**When Azure wins:** the client is M365/Entra-first — Entra Agent ID gives the native agent-identity story.

**When Google wins:** the client is GCP/Workspace-native, or cost pressure favors Gemini Flash economics.

This is exactly why the program ships the same capstone on alternate stacks (offline): the
architecture is portable; the stack is a client-fit decision.

## 5. Integration-topology selector

Three patterns. The selector scores them against the pilot's actual constraints (few systems,
synchronous request/reply, ship in 10 weeks) and returns a recommendation with a *revisit trigger*.

In [10]:
@dataclass
class TopologyContext:
    num_systems: int
    realtime_required: bool       # event-driven / push?
    governance_required: bool     # central monitoring, many teams?
    time_to_ship_weeks: int
    expected_growth: str          # "low" | "high"

def select_topology(ctx: TopologyContext):
    scores = {"P2P (point-to-point)": 0, "iPaaS (managed hub)": 0, "Event mesh (pub/sub)": 0}
    reasons = {k: [] for k in scores}

    # Few systems + ship fast -> P2P
    if ctx.num_systems <= 4:
        scores["P2P (point-to-point)"] += 2; reasons["P2P (point-to-point)"].append("few systems (<=4)")
    else:
        scores["iPaaS (managed hub)"] += 2; reasons["iPaaS (managed hub)"].append("many systems -> N^2 wiring")
    if ctx.time_to_ship_weeks <= 12:
        scores["P2P (point-to-point)"] += 2; reasons["P2P (point-to-point)"].append("tight ship deadline")
    # Real-time -> event mesh
    if ctx.realtime_required:
        scores["Event mesh (pub/sub)"] += 3; reasons["Event mesh (pub/sub)"].append("real-time / event-driven flows")
    else:
        scores["P2P (point-to-point)"] += 1; reasons["P2P (point-to-point)"].append("synchronous request/reply")
    # Governance -> iPaaS
    if ctx.governance_required:
        scores["iPaaS (managed hub)"] += 2; reasons["iPaaS (managed hub)"].append("central governance/monitoring needed")
    # Growth -> bias to iPaaS/event mesh
    if ctx.expected_growth == "high":
        scores["iPaaS (managed hub)"] += 1; reasons["iPaaS (managed hub)"].append("high growth ahead")
        scores["Event mesh (pub/sub)"] += 1; reasons["Event mesh (pub/sub)"].append("high growth ahead")

    pick = max(scores, key=scores.get)
    return pick, scores, reasons

# The Meridian pilot.
ctx = TopologyContext(num_systems=3, realtime_required=False, governance_required=False,
                      time_to_ship_weeks=10, expected_growth="high")
pick, sc, reasons = select_topology(ctx)

rule("INTEGRATION-TOPOLOGY SELECTOR — Meridian pilot")
for k in sc:
    why = "; ".join(reasons[k]) or "—"
    print(f"  {k:24} score={sc[k]}   ({why})")
print(f"\nRecommended: {pick}")
print("  Implementation note: P2P, but funneled through the ONE MCP server — so you still get a")
print("  single governed door (RBAC + audit) even inside a point-to-point topology.")
print("  Revisit trigger: when Meridian moves to event-driven dispute intake or many more")
print("  source systems, the event mesh earns its operational complexity.")

INTEGRATION-TOPOLOGY SELECTOR — Meridian pilot
  P2P (point-to-point)     score=5   (few systems (<=4); tight ship deadline; synchronous request/reply)
  iPaaS (managed hub)      score=1   (high growth ahead)
  Event mesh (pub/sub)     score=1   (high growth ahead)

Recommended: P2P (point-to-point)
  Implementation note: P2P, but funneled through the ONE MCP server — so you still get a
  single governed door (RBAC + audit) even inside a point-to-point topology.
  Revisit trigger: when Meridian moves to event-driven dispute intake or many more
  source systems, the event mesh earns its operational complexity.


In [11]:
# Show the selector flipping for a different future state (the expansion).
future = TopologyContext(num_systems=9, realtime_required=True, governance_required=True,
                         time_to_ship_weeks=26, expected_growth="high")
fpick, fsc, _ = select_topology(future)
print("Future state (9 systems, real-time intake, governance, 26wk):")
for k in fsc: print(f"  {k:24} score={fsc[k]}")
print(f"  -> {fpick}  (the pilot's P2P choice correctly does NOT survive expansion)")

Future state (9 systems, real-time intake, governance, 26wk):
  P2P (point-to-point)     score=0
  iPaaS (managed hub)      score=5
  Event mesh (pub/sub)     score=4
  -> iPaaS (managed hub)  (the pilot's P2P choice correctly does NOT survive expansion)


## 6. The procurement questionnaire — and the control each item wants

The class centerpiece runs six of Meridian's **15-item vendor security questionnaire** live (the full question text is in `persona-procurement-script.md`, sent as the pre-read). This is the take-home answer key: for every item, the control a *strong* vendor names. The rubric from the reveal applies to all 15 — a passing answer **names the control, names where it lives (the integration layer, not the LLM), and offers proof**. A failing answer is any variant of "we'll handle that during the build."

The six items interrogated live in class (#3, #4, #8, #9, #10, #15) are answered in depth; the rest get the one-line control a strong vendor leads with.

| # | Questionnaire item | The control a strong vendor names |
|---|---|---|
| 1 | Data residency & segregation | Single-tenant deployment inside *Meridian's own* AWS account and region; VPC-isolated; no client data ever leaves Meridian's cloud or co-tenants with another customer. |
| 2 | Data classification & minimization | A field-level data inventory in the SoW data annex: only the dispute/transaction fields the workflow needs, each one justified; everything else is never ingested. |
| **3** | **Synthetic vs. production data** *(live round)* | **Yes — synthetic-only until the controls review passes, and nothing breaks.** The build, tests, demos, and the entire eval suite run against a seeded synthetic dataset that mirrors production shape (same schemas, same reason codes, same Reg E clocks). Strong vendors treat this as an architecture property, not a concession: if the system can't be built on synthetic data, it's the wrong architecture for a bank. Production access is a dated milestone gated on the CISO's controls review. |
| **4** | **Identity & agent authorization** *(live round)* | **The agent acts under the analyst's identity via OAuth2 on-behalf-of — it never has standing access of its own.** Scope is enforced in the MCP integration layer, not the model, and every call is audited with the acting identity. An analyst scoped to Type-A disputes physically cannot pull Type-B records through the agent — and the vendor offers proof: the two-analyst test (same request, two identities, one 200 and one 403). |
| 5 | Access control & least privilege | RBAC enforced at the integration layer (the one governed MCP door) and honored by the system of record — never delegated to the prompt; proven end-to-end with the two-analyst 403/200 test. |
| 6 | Authentication & secrets management | A managed secrets store (e.g. AWS Secrets Manager) with rotation and instant revocation; no secret in source control, and never — under any circumstances — in a prompt. |
| 7 | Encryption | TLS in transit, KMS-managed encryption at rest, with *Meridian-owned* keys — so the bank can revoke the vendor's access unilaterally, without asking. |
| **8** | **Audit logging & traceability** *(live round)* | **An append-only audit log capturing every tool call, read, and write — with the acting identity, timestamp, and the allow/deny decision.** "Append-only" is the load-bearing word: neither the vendor nor the agent can rewrite history. The log is exportable to the bank's own SIEM on demand, so traceability doesn't depend on trusting the vendor's console. |
| **9** | **Prompt injection & untrusted content** *(live round)* | **An input guardrail on all attacker-influenceable text, with quarantine for poisoned content.** Documents and tickets are treated as untrusted data, never as instructions: they enter through the governed MCP door, get screened before reaching the model, and suspicious chunks are quarantined rather than silently dropped. Output-side, exfiltration is capped because the agent can only act through scoped, audited tools — there is no raw egress path for data to leak through. |
| **10** | **Model & output governance** *(live round)* | **A citation tripwire plus human-in-the-loop on low confidence.** Any regulated claim (a Reg E deadline, a provisional-credit rule) must carry a citation into the policy knowledge base; an answer that can't cite is blocked, not shipped. Low-confidence cases route to the analyst approval queue with evidence pre-gathered — so a confidently-wrong deadline is caught by a control, not by the customer. |
| 11 | Human-in-the-loop & autonomy limits | Money-moving and customer-facing actions always require human approval via the analyst queue; the boundary lives in change-controlled configuration, not in anyone's head. |
| 12 | Vulnerability management & SDLC | Dependency scanning and patch SLAs, plus the eval gate: a 150-case golden suite runs in CI and *blocks* any deploy that regresses — security review as a pipeline step, not a meeting. |
| 13 | Incident response & breach notification | A contractual breach-notification window (hours, not "promptly") and a named, reachable 2am contact — written into the SoW, not promised on the call. |
| 14 | Subprocessors & model providers | The complete data-path list (cloud, model provider, any vector/search service), each with its documented retention and training-use policy — e.g. model calls stay inside the client's cloud via Bedrock, with no training on client data. |
| **15** | **Exit, data deletion & IP** *(live round)* | **An IP annex plus a verified-deletion exit plan.** On termination: the bank's data is deleted *including derived artifacts* — prompts, fine-tunes, embeddings — with deletion verified, not just attested; all vendor access is revoked (the bank holds the keys, see #7); and the IP annex states in writing who owns what was built during the engagement. Strong vendors welcome this question — a clean exit story is what lets a nervous buyer sign. |

Notice the pattern across all 15: every strong answer lives in the **integration layer or the contract** — almost none of them live in the model. That is the week's thesis in one table: the deal is won on paper and architecture, and you will build every one of these controls in W15–W17.


## Glossary — every named term from the slides

**Estimation & commercial**
- **T-shirt sizing** — sizing work as S/M/L/XL day-*ranges* instead of false-precision hours.
- **Risk buffer** — a multiplier on an estimate's high end, applied *per risk class*; access-politics gets the biggest.
- **Milestone** — a shippable unit of the engagement with its own sign-off gate (M0–M4).
- **T&M-cap (Time & Materials with a cap)** — bill on hours, ceiling-bounded; risk shared, upside capped.
- **Fixed-price** — one number for a scoped deliverable; vendor owns all overrun risk.
- **Outcome-based pricing** — bill on a business result (e.g., a resolved dispute); aligned, high variance.
- **Acceptance criteria** — the measurable "done" in the SoW; here, the M0 eval anchors verbatim.
- **IP terms** — who owns agents, prompts, fine-tunes, embeddings on completion.
- **Data annex** — the SoW section on data: classes, residency, subprocessors, encryption, audit, deletion.
- **Change control** — how scope changes get priced and approved.
- **Risk register** — ranked risks with likelihood × impact, mitigation, owner.

**The live cases**
- **Sierra** — Bret Taylor / Clay Bavor's agent company; outcome-based pricing; reported $150M+ ARR.
- **Palantir acquire-expand-scale** — land small, embed, prove value, expand (Airbus → Skywise).
- **"Data is political" / 8–12 weeks of access politics per week of value** — Nabeel Qureshi, *Reflections on Palantir* (2023).

**Architecture & stack**
- **OpenAI Agents SDK** — the Python agent framework used in the build labs.
- **AWS Bedrock** — the managed model service; Claude runs inside the client's own AWS account (the models come to the client's cloud).
- **Bedrock Knowledge Base** — the managed RAG knowledge base over the policy docs; grounds regulated answers with citations.
- **MCP server (Model Context Protocol)** — the one governed door between agents and client systems; where RBAC + audit live.
- **OAuth2 on-behalf-of (OBO)** — token exchange so the agent acts under the *end user's* identity and scope.
- **AWS IAM / Identity Center** — identity for users; scoped agent keys give the agent least-privilege credentials.
- **AgentCore Identity** — a governed identity for the *agent* itself (Amazon Bedrock AgentCore).
- **Promptfoo** — the portable eval harness; runs continuously and as the CI deploy gate.
- **OpenTelemetry (OTel)** — tracing standard feeding the dashboards.
- **CloudWatch** — where OTel traces land; cost/latency/eval dashboards.
- **HITL (human-in-the-loop)** — the human-approval step required for money-moving actions.
- **Stack-choice decision matrix** — weighted scoring of candidate stacks against client fit.
- *Alternate stacks in the matrix:* **Microsoft Foundry** (Azure's agent platform, with native **Entra Agent ID** identity for agents) and **Google ADK + Gemini** — each wins when the client's estate is Azure- or GCP-first.

**Integration topology**
- **P2P (point-to-point)** — direct system-to-system connections; simple, N² at scale.
- **iPaaS** — integration-platform-as-a-service; a managed hub for many systems with governance.
- **Event mesh** — a pub/sub event backbone for real-time, decoupled flows.

In [12]:
print("Notebook complete. All five calculators ran. See lab/sow-template.md and")
print("lab/milestone-M1-sow-arch.md for the graded M1 artifacts.")

Notebook complete. All five calculators ran. See lab/sow-template.md and
lab/milestone-M1-sow-arch.md for the graded M1 artifacts.
